<a href="https://colab.research.google.com/github/saswanth01/MLA0304/blob/main/Lab/26_30_D.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

26

In [ ]:
import numpy as np

prices = [50, 60, 70, 80, 90]
prob = [0.30, 0.40, 0.50, 0.45, 0.35]
N = 1000

def bandit(method):
    count = np.zeros(5)
    success = np.zeros(5)
    revenue = 0

    for t in range(N):

        if method == "epsilon":
            if np.random.rand() < 0.1:
                a = np.random.randint(5)
            else:
                avg = np.divide(success, count, out=np.zeros(5), where=count>0)
                a = np.argmax(avg)

        elif method == "ucb":
            if t < 5:
                a = t
            else:
                avg = success / count
                ucb = avg + np.sqrt(2*np.log(t)/count)
                a = np.argmax(ucb)

        else:
            samples = [
                np.random.beta(success[i]+1, count[i]-success[i]+1)
                for i in range(5)
            ]
            a = np.argmax(samples)

        sale = np.random.rand() < prob[a]

        count[a] += 1
        if sale:
            success[a] += 1
            revenue += prices[a]

    return revenue

for method in ["epsilon", "ucb", "thompson"]:
    print(method, "Revenue =", round(bandit(method), 2))

27

In [ ]:
import random

roads = {
    "A": ["B", "C"],
    "B": ["D"],
    "C": ["D"],
    "D": ["E"],
    "E": []
}

def navigate(start, goal):
    current = start
    path = [current]

    while current != goal:
        next_nodes = roads[current]
        current = random.choice(next_nodes)
        path.append(current)

    return path

for i in range(5):
    path = navigate("A", "E")
    print("Path:", path)

Path: ['A', 'C', 'D', 'E']
Path: ['A', 'B', 'D', 'E']
Path: ['A', 'B', 'D', 'E']
Path: ['A', 'B', 'D', 'E']
Path: ['A', 'C', 'D', 'E']


28

In [ ]:
import numpy as np

grid = np.zeros((4,4))
goal = (3,3)
V = np.zeros((4,4))

for _ in range(50):
    newV = V.copy()
    for i in range(4):
        for j in range(4):
            if (i,j) == goal:
                continue

            moves = []
            for di,dj in [(1,0),(-1,0),(0,1),(0,-1)]:
                ni,nj = i+di,j+dj
                if 0 <= ni < 4 and 0 <= nj < 4:
                    moves.append(-1 + 0.9*V[ni,nj])

            newV[i,j] = max(moves)
    V = newV

print("Optimal State Values:")
print(np.round(V,2))

Optimal State Values:
[[-4.69 -4.1  -3.44 -2.71]
 [-4.1  -3.44 -2.71 -1.9 ]
 [-3.44 -2.71 -1.9  -1.  ]
 [-2.71 -1.9  -1.    0.  ]]


29

In [ ]:
import numpy as np

states = ["Low", "Medium", "High"]
actions = ["Short", "Long"]

reward = {
    ("Low","Short"): 5, ("Low","Long"): 3,
    ("Medium","Short"): 2, ("Medium","Long"): 6,
    ("High","Short"): 1, ("High","Long"): 8
}

policy = {s: "Short" for s in states}

for _ in range(10):
    V = {s: reward[(s, policy[s])] for s in states}

    stable = True
    for s in states:
        best = max(actions, key=lambda a: reward[(s,a)])
        if best != policy[s]:
            policy[s] = best
            stable = False

    if stable:
        break

print("Optimal Policy:")
for s in states:
    print(s, "->", policy[s])

Optimal Policy:
Low -> Short
Medium -> Long
High -> Long


30

In [ ]:
import numpy as np
import random
from collections import deque
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Dense

model = Sequential([
    Dense(24, activation="relu", input_shape=(4,)),
    Dense(24, activation="relu"),
    Dense(3, activation="linear")
])

model.compile(optimizer="adam", loss="mse")

memory = deque(maxlen=2000)
gamma = 0.95
epsilon = 1.0

for episode in range(20):
    state = np.random.rand(4)
    total_reward = 0

    for step in range(50):
        if random.random() < epsilon:
            action = random.randrange(3)
        else:
            action = np.argmax(model.predict(state.reshape(1,4), verbose=0))

        next_state = np.random.rand(4)
        reward = 1 if action == 1 else -1

        memory.append((state, action, reward, next_state))
        state = next_state
        total_reward += reward

        if len(memory) >= 32:
            batch = random.sample(memory, 32)

            for s,a,r,ns in batch:
                target = r + gamma * np.max(
                    model.predict(ns.reshape(1,4), verbose=0)
                )
                q = model.predict(s.reshape(1,4), verbose=0)
                q[0][a] = target
                model.fit(s.reshape(1,4), q, epochs=1, verbose=0)

    epsilon *= 0.95
    print("Episode:", episode+1, "Reward:", total_reward)

/usr/local/lib/python3.13/dist-packages/keras/src/layers/core/dense.py:106: UserWarning: Do not pass an `input_shape`/`input_dim` argument to a layer. When using Sequential models, prefer using an `Input(shape)` object as the first layer in the model instead.
  super().__init__(activity_regularizer=activity_regularizer, **kwargs)


Episode: 1 Reward: -14
